# Preparação e diagnóstico de John — Semana 5

**Objetivo:** produzir uma base por pagante para a segmentação e comparar uso e churn por plano, canal e período. Corte: 30/09/2026.

O notebook é um roteiro de estudo. O código reutilizável está em `src/preparar_diagnostico_s5.py`; dados individuais permanecem fora do Git.


## 1. Antes de calcular: definir quem entra

O retrato de assinaturas não informa sozinho quem estava ativo no início de cada mês. Reconstruímos intervalos pelos eventos e usamos o retrato apenas para os inícios anteriores à cobertura do log. Desligar renovação não encerra o período pago.


In [ ]:
from pathlib import Path
import runpy
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'dados/brutos').is_dir()
runpy.run_path(str(ROOT / 'src/preparar_diagnostico_s5.py'), run_name='__main__')


## 2. Conferir população e denominadores

`expostos` soma assinantes no início de cada mês. Na comparação anual, uma pessoa pode contribuir em vários meses: não é uma contagem de pessoas distintas. A média mensal simples e a taxa ponderada por exposição respondem a perguntas ligeiramente diferentes.


In [ ]:
OUT = ROOT / 'dados/saidas/s5_john'
for name in ['churn_por_ano', 'churn_jan_set_comparavel', 'churn_plano_jan_set']:
    print(name)
    display(pd.read_csv(OUT / f'{name}.csv'))


## 3. Separar piora das taxas e mudança de composição

A decomposição por plano divide a diferença agregada em uma parcela de mudanças nas taxas e outra de mudanças nas participações. É uma identidade aritmética, não um teste causal.


In [ ]:
display(pd.read_csv(OUT / 'decomposicao_plano_jan_set.csv'))


## 4. Olhar o uso antes do desfecho

Usamos o mês anterior para evitar que o próprio cancelamento reduza a exposição ao produto no mês em que ocorreu. Exigimos que a pessoa já fosse pagante no início do mês anterior. Isso melhora a ordem temporal, mas não elimina diferenças de perfil ou prova causalidade.


In [ ]:
display(pd.read_csv(OUT / 'churn_uso_anterior.csv'))
display(pd.read_csv(OUT / 'churn_uso_anterior_por_plano.csv'))


## 5. Entender o arquivo que Leonardo recebe

Uma linha por pagante no corte, 17 variáveis candidatas e `user_id` apenas como chave. Não entram rótulo futuro, intenção de cancelamento ou valores financeiros. Leonardo deve escolher variáveis, avaliar correlação e escalas e decidir o tratamento das ausências. Ausência de treino na cobertura de 2026 e distância de zero dias desde o último treino são informações diferentes.


In [ ]:
dictionary = pd.read_csv(OUT / 'dicionario_base_analitica.csv')
display(dictionary[dictionary.candidata_clusterizacao])


## Para verificar sua compreensão

Por que medir treinos no próprio mês do cancelamento pode exagerar a relação entre baixo uso e churn?

Para a redação final, leia `dados/saidas/s5_john/sintese_para_relatorio.md`. As hipóteses permanecem distintas de causas comprovadas.
